# 00 — Environment setup and reproducibility

**Purpose:** check that this computer is ready for the evaluation workflow. Think of the environment and lockfile as the laboratory equipment and its recorded settings.

| Item | Description |
|---|---|
| Inputs | Existing uv environment, `pyproject.toml`, `uv.lock`, `.python-version`, YAML configuration; optional local `.env` |
| Outputs | Readiness tables and a reproducibility summary in memory; no files or API requests |
| Parameters | `configs/notebooks.yaml`; default `execution.mode: demo` |

Start here, then follow notebooks 01–04. Every notebook starts independently. Do not paste credentials or real citation text into notebook source. Clear all outputs before committing; the repository hook checks this.

## 1. Open the project environment

In a terminal at the repository root, run `uv sync --locked --group notebooks`, then `uv run --locked --group notebooks jupyter lab`. These terminal commands install dependencies and launch Jupyter; this notebook does not install anything.

In an IDE, choose the Python interpreter inside this repository's `.venv`. A kernel is the Python process executing notebook cells. Restarting it clears variables; **Restart Kernel and Run All** should work with the default synthetic configuration.

In [ ]:
try:
    from jev_eval.environment import initialize_notebook
except ModuleNotFoundError:
    raise RuntimeError(
        "Select this project's .venv Python kernel. From the repository terminal, "
        "run: uv sync --locked --group notebooks"
    ) from None

project = initialize_notebook()

## 2. Check Python and dependencies

The table below shows whether the project environment and packages are available. Missing hosted credentials are normal in demo mode. If the environment row is false, switch to the `.venv` kernel. If a package is missing, run the setup command above and restart the kernel.

In [ ]:
from jev_eval.environment import check_environment

check_environment(project)

## 3. Understand configuration

YAML files are human-readable settings. Evaluation files choose the data and models; backend files choose how requests are sent; model files identify each model. The notebook settings control demonstrations, limits, and permissions. Paths are resolved from the repository root, even when the notebook is opened inside `notebooks/`.

The initial configuration uses invented states and an offline demo backend. Its responses are not predictions from JEV-9B or JEV-27B.

In [ ]:
import pandas as pd

pd.DataFrame(
    [
        {
            "evaluation": project.workflow.evaluation_config,
            "mode": project.workflow.execution.mode,
            "record_limit": project.workflow.execution.max_records,
            "logical_request_limit": project.workflow.execution.max_logical_requests,
            "paid_calls_enabled": project.workflow.execution.allow_paid_calls,
            "writes_enabled": project.workflow.output.allow_writes,
        }
    ]
)

## 4. Configure credentials locally, when needed

Copy `.env.example` to `.env` in the terminal and enter provider settings there. `.env` is ignored by Git. The initialization function loads it without overwriting variables already set in your environment. Restart the kernel after changing credentials. Never print the API key or the complete environment.

The readiness table reports only whether variables are set. No hosted request is made here. The Inference.sh provider is implemented for a single synthetic connection test in notebook 02; it defaults to a mock transport. Other generic hosted providers still require their own adapter. Inference.sh bulk execution is not enabled.

## 5. Record reproducibility information

The lockfile fixes dependency resolutions. The Git revision identifies the code, while a dirty working tree means there are uncommitted changes. A missing commit is expected before the first commit. Final research runs should use a recorded, reviewed revision.

A fixed seed makes local sampling repeatable. It cannot guarantee identical responses from a hosted model that the provider may update.

In [ ]:
from jev_eval.environment import reproducibility_summary

reproducibility_summary(project)

## 6. Next step

Proceed to `01_prepare_data.ipynb`. All reports above exist only in memory. Package functions will create private run manifests in notebook 03 only when writing is explicitly enabled.